In [8]:
here::i_am("snakemake/01_create_arrow.R")
source(here::here("settings.R"))

# I/O
io$output.directory <- file.path(io$basedir,"ArchR_test5")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final

Setting default number of Parallel threads to 1.



In [16]:
args = list()
args$sample = 'BGRGP1'
args$min_fragments = 100
args$min_tss_score = 2

In [17]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[100:200]
geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [18]:
addArchRChrPrefix(chrPrefix = FALSE)

ArchR is now disabling the requirement of chromosome prefix = 'chr'



In [19]:
geneAnnotation$genes

GRanges object with 1794 ranges and 2 metadata columns:
          seqnames        ranges strand |            gene_id             symbol
             <Rle>     <IRanges>  <Rle> |        <character>        <character>
     [1] chrUn0078   14186-37826      - | ENSOCUG00000035257 ENSOCUG00000035257
     [2] chrUn0078   46261-47188      - | ENSOCUG00000026527 ENSOCUG00000026527
     [3] chrUn0078   91793-96899      + | ENSOCUG00000034758 ENSOCUG00000034758
     [4] chrUn0078 106635-108039      + | ENSOCUG00000037893 ENSOCUG00000037893
     [5] chrUn0078 240914-241862      - | ENSOCUG00000030026 ENSOCUG00000030026
     ...       ...           ...    ... .                ...                ...
  [1790] chrUn0178 268869-270549      + | ENSOCUG00000033412 ENSOCUG00000033412
  [1791] chrUn0178 312451-325326      - | ENSOCUG00000000865               ARSL
  [1792] chrUn0178 340582-341913      - | ENSOCUG00000028016 ENSOCUG00000028016
  [1793] chrUn0178 421791-423600      + | ENSOCUG00000031685 ENS

In [20]:
# trim 2kb ends of geneAnnotation, otherwise gives error: 

exclude = GRanges(
    seqnames = Rle(rep(genomeAnnotation$chromSizes@seqnames@values,2)),
    ranges = IRanges(start = c(genomeAnnotation$chromSizes@ranges@start, 
                               genomeAnnotation$chromSizes@ranges@width-2000), 
                     end = c(genomeAnnotation$chromSizes@ranges@start + 2000, 
                             genomeAnnotation$chromSizes@ranges@width)))

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$TSS))
if(nrow(exclude_ranges)){
geneAnnotation$TSS = geneAnnotation$TSS[-exclude_ranges$subjectHits]
}

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$genes))
if(nrow(exclude_ranges)){
geneAnnotation$genes = geneAnnotation$genes[-exclude_ranges$subjectHits]
}
exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$exons))
if(nrow(exclude_ranges)){
geneAnnotation$exons = geneAnnotation$exons[-exclude_ranges$subjectHits]
}

In [21]:
geneAnnotation$TSS
geneAnnotation$genes
geneAnnotation$exons
genomeAnnotation$chromSizes 

GRanges object with 3015 ranges and 2 metadata columns:
          seqnames    ranges strand |     tx_id            tx_name
             <Rle> <IRanges>  <Rle> | <integer>        <character>
     [1] chrUn0078     91793      + |     44066 ENSOCUT00000021943
     [2] chrUn0078    106635      + |     44067 ENSOCUT00000008810
     [3] chrUn0078    304085      + |     44068 ENSOCUT00000050165
     [4] chrUn0078    315009      + |     44069 ENSOCUT00000044757
     [5] chrUn0078    337503      + |     44070 ENSOCUT00000029289
     ...       ...       ...    ... .       ...                ...
  [3011] chrUn0178    325325      - |     47098 ENSOCUT00000034796
  [3012] chrUn0178    325307      - |     47099 ENSOCUT00000000866
  [3013] chrUn0178    324997      - |     47100 ENSOCUT00000063424
  [3014] chrUn0178    341912      - |     47101 ENSOCUT00000024657
  [3015] chrUn0178    476558      - |     47102 ENSOCUT00000014523
  -------
  seqinfo: 1402 sequences from an unspecified genome; no seqlen

GRanges object with 1757 ranges and 2 metadata columns:
          seqnames        ranges strand |            gene_id             symbol
             <Rle>     <IRanges>  <Rle> |        <character>        <character>
     [1] chrUn0078   14186-37826      - | ENSOCUG00000035257 ENSOCUG00000035257
     [2] chrUn0078   46261-47188      - | ENSOCUG00000026527 ENSOCUG00000026527
     [3] chrUn0078   91793-96899      + | ENSOCUG00000034758 ENSOCUG00000034758
     [4] chrUn0078 106635-108039      + | ENSOCUG00000037893 ENSOCUG00000037893
     [5] chrUn0078 240914-241862      - | ENSOCUG00000030026 ENSOCUG00000030026
     ...       ...           ...    ... .                ...                ...
  [1753] chrUn0178 268869-270549      + | ENSOCUG00000033412 ENSOCUG00000033412
  [1754] chrUn0178 312451-325326      - | ENSOCUG00000000865               ARSL
  [1755] chrUn0178 340582-341913      - | ENSOCUG00000028016 ENSOCUG00000028016
  [1756] chrUn0178 421791-423600      + | ENSOCUG00000031685 ENS

GRanges object with 14165 ranges and 3 metadata columns:
           seqnames        ranges strand |   exon_id            gene_id
              <Rle>     <IRanges>  <Rle> | <integer>        <character>
      [1] chrUn0078   91793-92364      + |    201633 ENSOCUG00000034758
      [2] chrUn0078   95515-96899      + |    201634 ENSOCUG00000034758
      [3] chrUn0078 106635-107173      + |    201635 ENSOCUG00000037893
      [4] chrUn0078 107896-108039      + |    201636 ENSOCUG00000037893
      [5] chrUn0078 304085-305015      + |    201637 ENSOCUG00000036626
      ...       ...           ...    ... .       ...                ...
  [14161] chrUn0178 340582-340639      - |    215864 ENSOCUG00000028016
  [14162] chrUn0178 341454-341747      - |    215865 ENSOCUG00000028016
  [14163] chrUn0178 341783-341913      - |    215866 ENSOCUG00000028016
  [14164] chrUn0178 476220-476410      - |    215867 ENSOCUG00000021865
  [14165] chrUn0178 476521-476559      - |    215868 ENSOCUG00000021865
       

GRanges object with 101 ranges and 0 metadata columns:
         seqnames    ranges strand
            <Rle> <IRanges>  <Rle>
    [1] chrUn0078 1-1207509      *
    [2] chrUn0079 1-1187392      *
    [3] chrUn0080 1-1157137      *
    [4] chrUn0081 1-1166848      *
    [5] chrUn0082 1-1147153      *
    ...       ...       ...    ...
   [97] chrUn0174  1-700076      *
   [98] chrUn0175  1-571908      *
   [99] chrUn0176  1-557607      *
  [100] chrUn0177  1-597847      *
  [101] chrUn0178  1-577984      *
  -------
  seqinfo: 3242 sequences from an unspecified genome

In [22]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)

ArchR logging to : ArchRLogs/ArchR-createArrows-55461eed4733-Date-2022-01-28_Time-15-32-41.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-28 15:32:41 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-28 15:32:41 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0 mins elapsed.

2022-01-28 15:32:41 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:32:50 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 2 Percent, 0.152 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:33:00 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 4 Perc

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:37:39 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 63 Percent, 4.962 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:37:47 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 65 Percent, 5.094 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:37:56 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 67 Percent, 5.249 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-28 15:38:05 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 69 Percent, 5.405 mins elapsed.




************************************************************
2022-01-28 15:51:53 : ERROR Found in .fastFeatureCounts for (rabbit_BGRGP1 : 1 of 1) 
LogFile = ArchRLogs/ArchR-createArrows-55461eed4733-Date-2022-01-28_Time-15-32-41.log

<simpleError in .getFragsFromArrow(ArrowFile = ArrowFile, chr = names(featureList)[x],     out = "IRanges", cellNames = cellNames): Chromosome NA not in ArrowFile! Available Chromosomes are : chrUn0078,chrUn0079,chrUn0080,chrUn0081,chrUn0082,chrUn0083,chrUn0084,chrUn0085,chrUn0086,chrUn0087,chrUn0088,chrUn0089,chrUn0090,chrUn0091,chrUn0092,chrUn0093,chrUn0094,chrUn0095,chrUn0096,chrUn0097,chrUn0098,chrUn0099,chrUn0100,chrUn0101,chrUn0102,chrUn0103,chrUn0104,chrUn0105,chrUn0106,chrUn0107,chrUn0108,chrUn0109,chrUn0110,chrUn0111,chrUn0112,chrUn0113,chrUn0114,chrUn0115,chrUn0116,chrUn0117,chrUn0118,chrUn0119,chrUn0120,chrUn0121,chrUn0122,chrUn0123,chrUn0124,chrUn0125,chrUn0126,chrUn0127,chrUn0128,chrUn0129,chrUn0130,chrUn0131,chrUn0132,chrUn0133,chrUn0134,chr

createArrowFiles has encountered an error, checking if any ArrowFiles completed..

2022-01-28 15:51:53 : 

ArchR logging successful to : ArchRLogs/ArchR-createArrows-55461eed4733-Date-2022-01-28_Time-15-32-41.log



In [ ]:
# Calculate doublet scores

ArrowFile = paste0(io$output.directory, '/rabbit_', args$sample, '.arrow')

doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 15, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)
